# Butyric Acid

In [1]:
using Clapeyron, Metaheuristics, Printf

In [2]:
like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
butyric,88.11,3.44685,3.327741373,221.2086601,1,1
"""

assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
butyric,H,butyric,e,2000.0,0.03
"""

model = PCSAFT(["butyric"], userlocations = [like_parameter, assoc_parameter])

PCSAFT{BasicIdeal, Float64} with 1 component:
 "butyric"
Contains parameters: Mw, segment, sigma, epsilon, epsilon_assoc, bondvol

In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("satp_butyric.csv")
fix_line_endings("rhol_butyric.csv")

Fixed: satp_butyric.csv
Fixed: rhol_butyric.csv


In [4]:
### Saturation pressure — output in Pa (SI)
function my_saturation_p(model::EoSModel, T::Float64)
    sat = saturation_pressure(model, T)
    return sat[1]   # Pa
end

# Liquid density — output in kg/m³
function my_liquid_density(model::EoSModel, T::Float64)
    sat  = saturation_pressure(model, T)
    Mw   = model.params.Mw[1]      # g/mol
    rhol = 1.0 / sat[2]            # mol/m³
    return rhol * Mw / 1000.0      # mol/m³ × g/mol ÷ 1000 = kg/m³
end

my_liquid_density (generic function with 1 method)

In [5]:
toestimate = [
    Dict(
        :param   => :epsilon_assoc,
        :indices => (1,1),
        :lower   => 500.0,
        :upper   => 5000.0,
        :guess   => 2400.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => (1,1),
        :lower   => 0.001,
        :upper   => 0.50,
        :guess   => 0.4
    ),
]

2-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 5000.0, :param => :epsilon_assoc, :indices => (1, 1), :guess => 2400.0, :lower => 500.0)
 Dict(:upper => 0.5, :param => :bondvol, :indices => (1, 1), :guess => 0.4, :lower => 0.001)

In [6]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_butyric.csv",
        "satp_butyric.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))

Initial objective value: 0.2796800373577158


In [7]:
method = ECA(; options = Options(iterations = 10000, seed = 999))
 
params_opt, model_opt = optimize(objective, estimator, method)

([2761.69865963706, 0.06871633901368966], PCSAFT{BasicIdeal, Float64}("butyric"))

In [8]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [9]:
aard_p   = calculate_AAD(model_opt, "satp_butyric.csv",   my_saturation_p)


=== AAD: satp_butyric.csv ===

┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593



Clapeyron Estimator  exp           calc          ARD%    
380.4500    13198.680000  13136.898396  0.4681  
381.5500    13731.960000  13778.169683  0.3365  
382.9500    14678.532000  14632.836357  0.3113  
383.3500    14838.516000  14885.178027  0.3145  
387.2500    17478.252000  17546.307740  0.3894  
388.7500    18544.812000  18672.390727  0.6879  
391.4500    20904.576000  20854.344882  0.2403  
392.0500    21451.188000  21367.520179  0.3900  
395.9500    25224.144000  24971.063920  1.0033  
396.3500    25357.464000  25368.096394  0.0419  
402.6500    32396.760000  32360.820846  0.1109  
404.6500    34863.180000  34894.122607  0.0888  
407.6500    38876.112000  39005.137970  0.3319  
410.0500    42462.420000  42579.411608  0.2755  
AARD = 0.3565%


0.35645541813611936

In [10]:
aard_p   = calculate_AAD(model_opt, "rhol_butyric.csv",   my_liquid_density)


=== AAD: rhol_butyric.csv ===
Clapeyron Estimator  exp           calc          ARD%    
287.1500    963.800000    947.095434    1.7332  
293.1500    957.670000    942.004657    1.6358  
297.8500    953.400000    938.031002    1.6120  
298.1500    952.730000    937.777700    1.5694  
305.5500    945.700000    931.538763    1.4974  
313.9500    937.300000    924.467566    1.3691  
321.1500    930.200000    918.403293    1.2682  
331.9500    919.800000    909.277387    1.1440  
340.5500    911.300000    901.965063    1.0244  
AARD = 1.4282%


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


1.4281651129337576